# BAS-02 — pinned Blender golden scene + persistent recovery
CPU only. This notebook has not itself passed until both phases have evidence.
Run Phase A, save the generated folder path, then **Runtime → Disconnect and delete runtime**.
Start a fresh runtime and run setup + Phase B only. Do not rerun Phase A for recovery.
Google Drive mount requests access interactively; review the prompt yourself.
The gateway uses Python 3.12.14 via uv, independently of Colab kernel Python and Blender bpy Python.


## Setup — run in both the original and fresh runtime

In [ ]:
import pathlib, re, subprocess, sys
SOURCE_SHA = input("Reviewed BAS-02 PR head SHA (40 hex): ").strip()
assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_SHA)
REPO = pathlib.Path('/content/MCPBlender')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/nguyenvando420-spec/mcp-for-blender.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', SOURCE_SHA], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_SHA], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_SHA
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.13.5'], check=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'libxrender1', 'libxi6', 'libxfixes3', 'libxkbcommon0', 'libsm6', 'libgl1'], check=True)
subprocess.run(['uv', 'sync', '--frozen', '--python', '3.12.14'], cwd=REPO, check=True)
from google.colab import drive
drive.mount('/content/drive')


## Phase A — create and persist evidence

In [ ]:
import datetime, json, shutil
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + SOURCE_SHA[:12]
LOCAL = pathlib.Path('/content/bas02') / RUN_ID
subprocess.run(['uv', 'run', '--frozen', 'python', 'scripts/run_golden.py', '--output-dir', str(LOCAL)], cwd=REPO, check=True)
PERSISTED = pathlib.Path('/content/drive/MyDrive/BlenderAIStudio/BAS-02') / RUN_ID
assert not PERSISTED.exists(), 'Never overwrite prior evidence'
PERSISTED.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL, PERSISTED)
subprocess.run(['uv', 'run', '--frozen', 'python', 'scripts/run_golden.py', '--output-dir', str(PERSISTED), '--verify-only'], cwd=REPO, check=True)
print('SAVE THIS PATH FOR PHASE B:', PERSISTED)
print((PERSISTED / 'manifest.json').read_text())


## Stop here and destroy the runtime
Save this notebook and the folder path above. Use **Runtime → Disconnect and delete runtime**.
Reconnect to a fresh runtime; rerun Setup at the same commit. Run Phase B below without Phase A.
This is a manual acceptance gate: a same-runtime copy is not proof of persistence after shutdown.


## Phase B — recover from Drive after runtime deletion

In [ ]:
import datetime, hashlib, json, uuid
PERSISTED = pathlib.Path(input('Existing Phase A folder in /content/drive/MyDrive/BlenderAIStudio/BAS-02/: ').strip())
BASE = pathlib.Path('/content/drive/MyDrive/BlenderAIStudio/BAS-02').resolve()
assert PERSISTED.resolve().is_relative_to(BASE)
assert not pathlib.Path('/content/bas02').exists(), 'Use a fresh runtime; local Phase A state must be gone'
manifest = json.loads((PERSISTED / 'manifest.json').read_text())
assert manifest['source_commit'] == SOURCE_SHA and not manifest['source_dirty']
command = ['uv', 'run', '--frozen', 'python', 'scripts/run_golden.py', '--output-dir', str(PERSISTED), '--reopen']
result = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
result.check_returncode()
assert 'BAS02_RECOVERY_OK' in result.stdout
receipt = {'recovered_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
           'source_commit': SOURCE_SHA, 'bundle_manifest_sha256': hashlib.sha256((PERSISTED / 'manifest.json').read_bytes()).hexdigest(),
           'original_bundle_unchanged': True, 'phase_a_local_folder_absent': True,
           'manual_runtime_deletion_required': True, 'log': result.stdout + result.stderr}
receipt_path = BASE / ('recovery-' + uuid.uuid4().hex + '.json')
receipt_path.write_text(json.dumps(receipt, indent=2) + '\n')
print('BAS02_COLAB_RECOVERY_OK', receipt_path)


Submit the Phase A manifest/BLEND/PNG/logs, Phase B recovery receipt and notebook link.
Record the runtime-deletion action separately in the acceptance notes. Windows, Linux ARM64,
macOS x64 and GPU rendering are not qualified by this CPU notebook.
